In [1]:
import threading
import time
import socket

from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")
btns = base.btns_gpio

In [2]:
%%microblaze base.PMODB

#include "gpio.h"
#include "pyprintf.h"

//Function to turn on/off a selected pin of PMODB
int write_gpio(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}

void clear_gpios(){
    for(int i = 0; i < 8; ++i){
        write_gpio(i,0);
    }
}

In [3]:
def write_active_buzzer():
    # buzz for 0.5s
    write_gpio(3,1)
    time.sleep(0.5)
    write_gpio(3,0)

def write_passive_buzzer(freq):
    period = 1/freq
    start = time.time()
    # buzz for 0.5s
    while time.time() - start < 0.5:
        write_gpio(3, 1)
        time.sleep(period/2)
        write_gpio(3, 0)
        time.sleep(period/2)
        
def wait_button(n):
    while not btns[n].read():
        time.sleep(0.05)
    print("Button pressed. Client can begin")

In [4]:
clear_gpios()
write_active_buzzer()


In [8]:
# SERVER
def server_t(port):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    sock.bind(('0.0.0.0', port))

    sock.listen(1)

    conn, addr = sock.accept()
    print(f"Accepted connection from {addr}:{port}")

    while True:
        data = conn.recv(1024)
        if not data: # client disconnects
            break
        else:
            print(f"received message")
            write_active_buzzer()

In [6]:
# CLIENT
def client_t(port, ip):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    sock.connect((ip, port))
    print(f"Connected to {ip}:{port}")
    
    while True:
        if btns[1].read():
            time.sleep(0.15)
            sock.sendall(b"BEEP")
        if btns[2].read():
            break;

    sock.close()

In [11]:
client_port = 9999
server_port = 8888
ip_addr = "192.168.0.225"

threads = []

t = threading.Thread(target=server_t, args=(server_port,))
threads.append(t)
t.start()
print(f"Starting server")

t = threading.Thread(target=client_t, args=(client_port, ip_addr))
threads.append(t)
print(f"waiting to start... Press btn0")
wait_button(0)
t.start()
print(f"Starting client")

for t in threads:
    name = t.name
    t.join()
    print('{} joined'.format(name))

Starting server
waiting to start... Press btn0
Button pressed. Client can begin
Starting client
Connected to 192.168.0.225:9999
Accepted connection from ('192.168.0.225', 59384):8888
received message
received message
received message
received message
received message
received message
received message
Thread-11 (server_t) joined
Thread-12 (client_t) joined
